# 04 TabPFN

Same folds, features, target and output format as notebook 03. No tuning. Run on Colab with a GPU (Runtime > Change runtime type > T4 GPU).

- Input: `data/processed/long.csv` (and `model_table.csv` if notebook 03 has written it)
- Output: `results/predictions/04/`, `results/metrics/04_tabpfn_*.csv`

In [4]:
%pip install -q tabpfn==9.2.0

## 1. Access and paths

Store the Prior Labs key as prompted. Edit `ROOT` if the project sits elsewhere on Drive.

In [2]:
import os
import sys
from getpass import getpass
from pathlib import Path

from google.colab import drive
drive.mount("/content/drive")

os.environ["TABPFN_TOKEN"] = getpass("Prior Labs API key: ")
os.environ["TABPFN_NO_BROWSER"] = "1"

ROOT = Path("/content/drive/MyDrive/moringa-jar-test")
assert (ROOT / "src" / "config.py").exists(), "Project folder not found, edit ROOT"
sys.path.insert(0, str(ROOT))
print("project found")

Prior Labs API key: ··········
project found


In [5]:
import time
import warnings

import numpy as np
import pandas as pd
import torch
import tabpfn
from tabpfn import TabPFNRegressor

from src.config import DATA_PROC, RESULTS, SEED
from src.features import make_table
from src.cv import lodo, random_kfold, fold_ids
from src.metrics import scores

warnings.filterwarnings("ignore")
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"tabpfn {tabpfn.__version__} | device: {DEVICE}")

FEATURE_SETS = {"core": ["dose", "log_t0_raw", "sed"],
                "full": ["dose", "log_t0_raw", "sed", "speed", "mtime"]}
RUNS = [("core", "lodo"), ("core", "random"), ("full", "lodo")]
TARGET = "log_ratio"
QUANTILES = [0.05, 0.5, 0.95]

PRED_DIR = RESULTS / "predictions" / "04"
MET_DIR = RESULTS / "metrics"
for p in (PRED_DIR, MET_DIR):
    p.mkdir(parents=True, exist_ok=True)

tabpfn 9.2.0 | device: cuda


## 2. Model table and folds

In [6]:
long = pd.read_csv(DATA_PROC / "long.csv", parse_dates=["date"])
tab = make_table(long, level="cell")
groups = tab["day"].to_numpy()
tab["fold_lodo"] = fold_ids(lodo(groups), len(tab))
tab["fold_random"] = fold_ids(random_kfold(len(tab), 10), len(tab))

saved = DATA_PROC / "model_table.csv"
if saved.exists():
    ref = pd.read_csv(saved)
    same = len(ref) == len(tab) and all((ref[c].to_numpy() == tab[c].to_numpy()).all()
                                        for c in ["run", "sed", "fold_lodo", "fold_random"])
    assert same, "folds differ from notebook 03"
    print("folds match notebook 03")
else:
    print("model_table.csv not found, folds rebuilt with the same code as notebook 03")
print(tab.shape, "rows,", tab["day"].nunique(), "days")

folds match notebook 03
(1008, 17) rows, 22 days


In [7]:
def tabpfn_model():
    return TabPFNRegressor(device=DEVICE, random_state=SEED, ignore_pretraining_limits=True)


def predict_with_quantiles(m, X):
    mean = m.predict(X)
    try:
        q = np.column_stack(m.predict(X, output_type="quantiles", quantiles=QUANTILES))
    except Exception:
        q = np.full((len(X), len(QUANTILES)), np.nan)
    return mean, q

## 3. Run

In [8]:
y = tab[TARGET].to_numpy()
offset = tab["log_t0_raw"].to_numpy()
t0 = time.time()
for fs, split in RUNS:
    path = PRED_DIR / f"tabpfn__{fs}__{split}.csv"
    if path.exists():
        print(f"{fs} {split}: cached")
        continue
    X = tab[FEATURE_SETS[fs]].to_numpy(float)
    f = tab[f"fold_{split}"].to_numpy()
    pred = np.full(len(y), np.nan)
    q = np.full((len(y), len(QUANTILES)), np.nan)
    for k in np.unique(f):
        tr, te = f != k, f == k
        m = tabpfn_model().fit(X[tr], y[tr])
        pred[te], q[te] = predict_with_quantiles(m, X[te])
    out = tab[["run", "day", "sed", "t0_raw", "log_turb"]].assign(
        model="tabpfn", fset=fs, split=split, fold=f, pred=pred + offset, std=np.nan)
    for j, qq in enumerate(QUANTILES):
        out[f"q{int(qq * 100):02d}"] = q[:, j] + offset
    out.to_csv(path, index=False)
    print(f"{fs} {split}: done ({time.time() - t0:.0f} s)")

config.json:   0%|          | 0.00/36.0 [00:00<?, ?B/s]

core lodo: done (53 s)
core random: done (72 s)
full lodo: done (113 s)


## 4. Scores

In [9]:
def removal(log_t, t0):
    return 100 * (1 - 10 ** log_t / t0)

preds = pd.concat([pd.read_csv(p) for p in sorted(PRED_DIR.glob("tabpfn__*.csv"))], ignore_index=True)
rows, by_day = [], []
for (fs, sp), g in preds.groupby(["fset", "split"]):
    s = scores(g["log_turb"], g["pred"])
    day_rmse = np.sqrt(g.assign(se=(g["pred"] - g["log_turb"]) ** 2).groupby("day")["se"].mean())
    g60 = g[g["sed"] == 60]
    s.update({"model": "tabpfn", "fset": fs, "split": sp,
              "day_rmse_mean": day_rmse.mean(), "day_rmse_sd": day_rmse.std(),
              "rmse_log_60": np.sqrt(np.mean((g60["pred"] - g60["log_turb"]) ** 2)),
              "mae_removal60_pp": np.mean(np.abs(removal(g60["pred"], g60["t0_raw"]) - removal(g60["log_turb"], g60["t0_raw"])))})
    if g["q05"].notna().all():
        s["coverage_90"] = float(((g["log_turb"] >= g["q05"]) & (g["log_turb"] <= g["q95"])).mean())
        s["width_90_log"] = float((g["q95"] - g["q05"]).mean())
    rows.append(s)
    by_day.append(day_rmse.rename("rmse").reset_index().assign(model="tabpfn", fset=fs, split=sp))

score_tab = pd.DataFrame(rows).set_index(["fset", "split", "model"]).sort_index()
pd.concat(by_day, ignore_index=True).to_csv(MET_DIR / "04_tabpfn_scores_by_day.csv", index=False)
score_tab.round(4).to_csv(MET_DIR / "04_tabpfn_scores.csv")
score_tab.round(4)

rmse_log  mae_log  r2_log  rmse_ntu     mape  \
fset split  model                                                  
core lodo   tabpfn    0.1762   0.1383  0.6055   21.9480  33.1059   
     random tabpfn    0.0235   0.0174  0.9930    3.3736   4.0292   
full lodo   tabpfn    0.1364   0.1023  0.7637   17.2328  24.3528   

                    day_rmse_mean  day_rmse_sd  rmse_log_60  mae_removal60_pp  \
fset split  model                                                               
core lodo   tabpfn         0.1592       0.0808       0.2099            8.7637   
     random tabpfn         0.0231       0.0077       0.0224            1.0143   
full lodo   tabpfn         0.1215       0.0691       0.1693            7.6424   

                    coverage_90  width_90_log  
fset split  model                              
core lodo   tabpfn       0.8016        0.5036  
     random tabpfn       0.9107        0.0741  
full lodo   tabpfn       0.8601        0.4855